EXERCÍCIO 1: Regressão Logística (Chatbot Bancário)


In [4]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

dados_treino = [
    ("quero consultar meu saldo", "consultar_saldo"),
    ("qual é o meu saldo", "consultar_saldo"),
    ("quanto tenho na conta", "consultar_saldo"),
    ("ver saldo da conta", "consultar_saldo"),
    ("mostrar meu saldo", "consultar_saldo"),

    ("quero transferir 100 reais", "transferir"),
    ("transferir 200 reais", "transferir"),
    ("quero fazer uma transferência", "transferir"),
    ("enviar 50 reais", "transferir"),
    ("mandar 300 reais para outra conta", "transferir"),

    ("quero pagar um boleto", "pagar_boleto"),
    ("pagar boleto de 150 reais", "pagar_boleto"),
    ("preciso pagar uma conta", "pagar_boleto"),
    ("pagar conta de 80 reais", "pagar_boleto"),
    ("realizar pagamento de boleto", "pagar_boleto"),

    ("quero fazer um depósito", "depositar"),
    ("depositar 500 reais", "depositar"),
    ("colocar 100 reais na conta", "depositar"),
    ("fazer depósito de 250 reais", "depositar"),
    ("adicionar dinheiro na conta", "depositar"),

    ("qual a previsão do tempo", "fora_escopo"),
    ("quem ganhou o jogo", "fora_escopo"),
    ("me conte uma piada", "fora_escopo"),
    ("qual é a capital do brasil", "fora_escopo"),
    ("quero pedir uma pizza", "fora_escopo")
]

stopwords = [
    "a", "o", "as", "os", "um", "uma",
    "de", "da", "do", "das", "dos",
    "para", "por", "na", "no", "nas", "nos",
    "meu", "minha", "me", "eu"
]

# 1. PRÉ-PROCESSAMENTO
def pré_processar(texto):
    texto_limpo = re.sub(r'[^\w\s]', '', texto.lower()).strip()
    tokens = texto_limpo.split()

    # TODO 1.1
    tokens_filtrados = [token for token in tokens if token not in stopwords]

    return " ".join(tokens_filtrados)


X_treino_limpo = [pré_processar(item[0]) for item in dados_treino]
y_treino = [item[1] for item in dados_treino]


# 2. VETORIZAÇÃO E TREINAMENTO
vectorizer = TfidfVectorizer()

# TODO 1.2
X_vetorizado = vectorizer.fit_transform(X_treino_limpo)

modelo = LogisticRegression()
modelo.fit(X_vetorizado, y_treino)


# 3. EXTRAÇÃO DE ENTIDADE (REGEX)
def extrair_valor_reais(texto_original):
    match = re.search(r'(\d+)\s*reais', texto_original, re.IGNORECASE)

    # TODO 1.3
    return float(match.group(1)) if match else None


# 4. PIPELINE NLU
def processar_nlu(mensagem_usuario, threshold=0.55):
    texto_p = pré_processar(mensagem_usuario)
    vetor_input = vectorizer.transform([texto_p])

    probas = modelo.predict_proba(vetor_input)[0]

    # TODO 1.4
    maior_confianca = max(probas)
    intencao_prevista = modelo.classes_[probas.argmax()]

    valor = extrair_valor_reais(mensagem_usuario)

    # TODO 1.5
    if maior_confianca < threshold or intencao_prevista == "fora_escopo":
        return {
            "status": "FALLBACK",
            "mensagem": "Não consegui entender sua solicitação.",
            "confianca": round(maior_confianca, 4)
        }

    else:
        return {
            "status": "SUCESSO",
            "intencao": intencao_prevista,
            "confianca": round(maior_confianca, 4),
            "valor": valor
        }


# --- TESTE 1 ---
if __name__ == "__main__":
    print(processar_nlu("quero transferir 200 reais"))

{'status': 'FALLBACK', 'mensagem': 'Não consegui entender sua solicitação.', 'confianca': np.float64(0.4385)}


EXERCÍCIO 2: Naive Bayes (Filtro de Spam / Triagem)

In [5]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

dados_treino = [
    ("Urgente: o servidor caiu e a aplicacao esta fora", "incidente_critico"),
    ("Erro grave no banco de dados e sistema parou", "incidente_critico"),
    ("Como faco para alterar minha senha do e-mail", "duvida_suporte"),
    ("Qual o procedimento para solicitar novo mouse", "duvida_suporte"),
    ("Gostaria de saber o horario de funcionamento", "fora_escopo")
]

stopwords = ["o", "a", "e", "do", "da", "para", "de"]

# 1. PRÉ-PROCESSAMENTO
def pré_processar(texto):
    texto_limpo = re.sub(r'[^\w\s]', '', texto.lower()).strip()
    tokens = texto_limpo.split()

    # TODO 2.1
    tokens_filtrados = [token for token in tokens if token not in stopwords]

    return " ".join(tokens_filtrados)


X_treino_limpo = [pré_processar(item[0]) for item in dados_treino]
y_treino = [item[1] for item in dados_treino]


# 2. VETORIZAÇÃO E TREINAMENTO
vectorizer = TfidfVectorizer()
X_vetorizado = vectorizer.fit_transform(X_treino_limpo)

# TODO 2.2
modelo_nb = MultinomialNB()

# TODO 2.3
modelo_nb.fit(X_vetorizado, y_treino)


# 3. EXTRAÇÃO DE ENTIDADE
def extrair_protocolo(texto):
    match = re.search(r'INC-?\d{4}', texto, re.IGNORECASE)

    # TODO 2.4
    return match.group(0).upper() if match else None


# 4. PIPELINE NLU
def processar_triagem(mensagem_usuario, threshold=0.60):
    texto_p = pré_processar(mensagem_usuario)
    vetor_input = vectorizer.transform([texto_p])

    probas = modelo_nb.predict_proba(vetor_input)[0]

    maior_confianca = max(probas)
    intencao_prevista = modelo_nb.classes_[probas.argmax()]

    protocolo = extrair_protocolo(mensagem_usuario)

    # TODO 2.5
    if maior_confianca < threshold:
        return {
            "status": "FALLBACK",
            "mensagem": "Não foi possível classificar o chamado com confiança suficiente.",
            "confianca": round(float(maior_confianca), 4)
        }
    else:
        return {
            "status": "SUCESSO",
            "intencao": intencao_prevista,
            "confianca": round(float(maior_confianca), 4),
            "protocolo": protocolo
        }


# --- TESTE 2 ---
if __name__ == "__main__":
    print(processar_triagem("Urgente: servidor caiu no protocolo INC-9982"))

{'status': 'FALLBACK', 'mensagem': 'Não foi possível classificar o chamado com confiança suficiente.', 'confianca': 0.5456}


EXERCÍCIO 3: Árvore de Decisão (Chatbot de E-Commerce)

In [6]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier

dados_treino = [
    ("Onde esta meu pedido e qual o rastreio", "rastrear_pedido"),
    ("Quero saber a localizacao da minha encomenda", "rastrear_pedido"),
    ("Quero cancelar minha compra e pedir reembolso", "cancelar_compra"),
    ("Preciso cancelar o pedido feito ontem", "cancelar_compra"),
    ("Qual o horario de atendimento da loja fisica", "fora_escopo")
]

stopwords = ["o", "a", "meu", "minha", "e", "da", "do"]

# 1. PRÉ-PROCESSAMENTO
def pré_processar(texto):
    texto_limpo = re.sub(r'[^\w\s]', '', texto.lower()).strip()
    tokens = texto_limpo.split()

    # TODO 3.1
    tokens_filtrados = [token for token in tokens if token not in stopwords]

    return " ".join(tokens_filtrados)


X_treino_limpo = [pré_processar(item[0]) for item in dados_treino]
y_treino = [item[1] for item in dados_treino]


# 2. VETORIZAÇÃO E TREINAMENTO
vectorizer = TfidfVectorizer()
X_vetorizado = vectorizer.fit_transform(X_treino_limpo)

# TODO 3.2
modelo_tree = DecisionTreeClassifier(random_state=42)

# TODO 3.3
modelo_tree.fit(X_vetorizado, y_treino)


# 3. EXTRAÇÃO DE ENTIDADE
def extrair_codigo_rastreio(texto):
    match = re.search(r'BR\d{9}', texto, re.IGNORECASE)

    # TODO 3.4
    return match.group(0).upper() if match else None


# 4. PIPELINE NLU
def processar_ecommerce(mensagem_usuario, threshold=0.50):
    texto_p = pré_processar(mensagem_usuario)
    vetor_input = vectorizer.transform([texto_p])

    probas = modelo_tree.predict_proba(vetor_input)[0]

    maior_confianca = max(probas)
    intencao_prevista = modelo_tree.classes_[probas.argmax()]

    codigo_rastreio = extrair_codigo_rastreio(mensagem_usuario)

    # TODO 3.5
    if maior_confianca < threshold or intencao_prevista == "fora_escopo":
        return {
            "status": "FALLBACK",
            "mensagem": "Não foi possível identificar a solicitação.",
            "confianca": round(float(maior_confianca), 4)
        }
    else:
        return {
            "status": "SUCESSO",
            "intencao": intencao_prevista,
            "confianca": round(float(maior_confianca), 4),
            "codigo_rastreio": codigo_rastreio
        }


# --- TESTE 3 ---
if __name__ == "__main__":
    print(processar_ecommerce("Quero rastrear o pedido BR987654321"))

{'status': 'SUCESSO', 'intencao': np.str_('rastrear_pedido'), 'confianca': 1.0, 'codigo_rastreio': 'BR987654321'}
